# Anomaly Detection & Risk Identification

This notebook demonstrates automated KPI monitoring using both static threshold-based rules and dynamic statistical Z-score detection. We evaluate 30-day lookback windows, classify anomaly severity, persist an audit log trail, and plot time-series metrics with shaded confidence bands.

## Setup & Loading Data

In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath(".."))
from scripts.anomaly_detection import (
    load_data,
    check_thresholds,
    detect_anomalies_zscore,
    classify_severity,
    log_anomalies,
    visualize_anomalies,
    ALERT_RULES
)

RAW_DATA_PATH = "../data/raw/anomaly_monitoring_data.csv"
df = load_data(RAW_DATA_PATH)
print(f"Loaded {len(df)} daily monitoring records.")
df.tail()

## Task 1: Threshold-Based Alerting

Static business rules flag metrics that exceed known Operational Level Agreements (OLAs). We evaluate current daily metrics against defined min/max rules:

In [ ]:
latest_metrics = {
    'daily_revenue': float(df.iloc[-1]['daily_revenue']),
    'transaction_count': float(df.iloc[-1]['transaction_count']),
    'signup_rate': float(df.iloc[-1]['signup_rate'])
}

alerts = check_thresholds(latest_metrics, ALERT_RULES)
print("=== THRESHOLD ALERTS ===")
for alert in alerts:
    print(f"⚠️ {alert['metric']} {alert['direction']}: {alert['value']} (Threshold: {alert['threshold']})")
if not alerts:
    print("✓ All metrics within defined threshold rules.")

## Task 2 & 3: Statistical Z-Score Detection & Severity Classification

Statistical monitoring calculates rolling mean $\mu$ and standard deviation $\sigma$ to compute Z-scores:
$$Z = \frac{|X - \mu|}{\sigma}$$

Anomalies with $Z > 2.0$ are flagged and classified by impact:
- **CRITICAL**: $Z > 3.0$
- **HIGH**: $Z > 2.0$
- **MEDIUM**: $Z > 1.5$
- **LOW**: $Z \le 1.5$

In [ ]:
series = df.set_index('date')['daily_revenue'].tail(30)
anomalies, z_scores = detect_anomalies_zscore(series, threshold=2.0)
mean = series.mean()
std = series.std()

severity_records = []
for date_val, val in anomalies.items():
    sev = classify_severity(val, mean, std)
    date_str = pd.to_datetime(date_val).strftime('%Y-%m-%d')
    severity_records.append({
        'date': date_str,
        'value': val,
        'z_score': z_scores[date_val],
        'severity': sev
    })

sev_df = pd.DataFrame(severity_records)
print("=== STATISTICAL ANOMALIES DETECTED ===")
print(sev_df)

critical = sev_df[sev_df['severity'].isin(['CRITICAL', 'HIGH'])]
print(f"\n⚠️ {len(critical)} Critical/High severity anomalies require immediate investigation.")

## Task 4: Anomaly Logging & Audit Trail

All flagged anomalies are logged to persistent storage (`../output/anomalies_log.csv`) for incident response tracking.

In [ ]:
audit_log_df = log_anomalies(df, metric_col='daily_revenue', output_csv="../output/anomalies_log.csv")
print("=== PERSISTED ANOMALY AUDIT TRAIL ===")
audit_log_df

## Task 5: Time-Series Visualization

We plot raw daily revenue over time, overlaying the 7-day rolling average, shading the $\pm 2\sigma$ expected range, and highlighting flagged anomalies with red 'X' markers.

In [ ]:
visualize_anomalies(df, metric_col='daily_revenue', output_path="../output/anomaly_detection.png")